<a href="https://colab.research.google.com/github/codebysumit/cryptography-algorithms/blob/master/notebooks/rail_fence_cipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rail Fence Cipher

## History
The Rail Fence Cipher is one of the oldest and simplest **transposition ciphers**. It was used as far back as ancient Greece, and later during the American Civil War for quick field communication. It does not need any complicated math, which made it easy to use by hand, but that same simplicity also makes it one of the weakest ciphers in this course.

## What is Rail Fence Cipher?
Every cipher we have covered so far, Caesar, Affine, Vigenere, One-Time Pad, Hill, Playfair, is a **substitution cipher**. A substitution cipher changes what each character actually is. The Rail Fence Cipher works completely differently, it is a **transposition cipher**. It never changes any character. It only changes the **order** the characters appear in.

The idea is to write the plaintext in a zigzag pattern across a number of imaginary rows, called **rails**, like a fence. Then the ciphertext is formed by reading the rails back, one full row at a time, left to right.

In this implementation, we support the full **printable ASCII range**, from space (` `) to tilde (`~`), which is ASCII value 32 to 126. Since this cipher only rearranges characters instead of transforming them, every character in that range, letters, digits, spaces, and punctuation, is treated exactly the same way. The **number of rails (the track size) is a variable key**, not fixed, so the same plaintext can produce many different ciphertexts depending on how many rails are chosen.

## Cryptography Algorithm

### Constants
*   $S = 32$ (Start of printable ASCII range)
*   $E = 126$ (End of printable ASCII range)
*   $r$ = Number of rails (the key), must be $2 \le r \le L$ where $L$ is the plaintext length

### 1. Building the Zigzag Pattern
Imagine writing each character of the plaintext down onto one of $r$ horizontal rails, moving down the rails one at a time, and then bouncing back up once you hit the last rail, like a ball bouncing between a floor and a ceiling.

*   Start at rail 0, moving downward.
*   Every character moves to the next rail in the current direction.
*   When rail 0 is reached, the direction switches to downward.
*   When rail $r-1$ (the last rail) is reached, the direction switches to upward.

This creates a rail number for every position in the plaintext, for example with 3 rails, the pattern of rail numbers repeats as: 0, 1, 2, 1, 0, 1, 2, 1, 0, ...

### 2. Encryption
Every character is placed onto the rail given by the zigzag pattern. Once every character has been placed, the ciphertext is formed by reading the rails in order, rail 0 first, then rail 1, then rail 2, and so on, each rail read left to right.

### 3. Decryption
Decryption works backward. First, the same zigzag pattern is rebuilt using the length of the ciphertext and the number of rails. This tells us exactly how many characters belong to each rail, and in what order they need to be placed back. The ciphertext is then sliced into chunks matching those rail sizes, and the characters are placed back into their original zigzag positions.

### 4. Fully Worked Example (By Hand)
Let's encrypt **WEAREDISCOVERED** using **3 rails**.

**Step 1: Work out the rail number for every letter**, bouncing between rail 0 and rail 2:

```
Letter: W  E  A  R  E  D  I  S  C  O  V  E  R  E  D
Rail:   0  1  2  1  0  1  2  1  0  1  2  1  0  1  2
```

**Step 2: Group the letters by rail.**

```
Rail 0: W . . . E . . . C . . . R . .
Rail 1: . E . R . D . S . O . E . E .
Rail 2: . . A . . . I . . . V . . . D
```

*   Rail 0 collects: **W E C R**
*   Rail 1 collects: **E R D S O E E**
*   Rail 2 collects: **A I V D**

**Step 3: Read the rails in order, top to bottom.**

**Ciphertext = WECRERDSOEEAIVD**

To decrypt, we would rebuild the same zigzag pattern, split the 15 character ciphertext into groups of 4, 7, and 4 (matching how many letters landed on each rail), and place them back into the zigzag positions in order.

### Key Requirements
*   The key is the **number of rails**, an integer.
*   It must be **at least 2**. A single rail does not scramble anything at all.
*   It cannot be **greater than the length of the text**, since a rail with no characters on it makes no sense.
*   More rails generally means a closer to diagonal zigzag with less noticeable scrambling, while very few rails create a stronger scramble. In fact, using $r = 2$ rails usually shuffles the text the most for this cipher.

### Starting Direction: Top or Bottom
There is a second, smaller option that changes the zigzag pattern: **where the zigzag begins**.

*   **From Top (left)**: the zigzag starts at rail 0 (the top rail) and moves downward first. This is the normal, default behaviour used in the worked example above.
*   **From Bottom (left)**: the zigzag starts at the very last rail (the bottom rail) and moves upward first.

Both directions use the exact same up-and-down bouncing logic, they just begin at opposite ends of the fence. This gives a second small key choice on top of the rail count, and both the encryption and decryption side must agree on which direction was used, or the message will not decrypt correctly.

### 1. Import Dependencies

In [ ]:
import random

### 2. Helper Utilities

In [ ]:
START_ASCII = 32
END_ASCII = 126

def validate_printable_text(text: str) -> None:
    # Rail Fence only rearranges characters, but every character must still fall
    # inside our supported printable ASCII range
    for ch in text:
        code = ord(ch)
        if not (START_ASCII <= code <= END_ASCII):
            raise ValueError(
                f"Character {ch!r} (ASCII {code}) is outside the supported range "
                f"{START_ASCII}-{END_ASCII}."
            )

def build_rail_pattern(length: int, num_rails: int, start_from: str = "top") -> list:
    # returns the rail number (row index) that each position 0..length-1 lands on
    # start_from='top' begins at rail 0 and moves downward first
    # start_from='bottom' begins at the last rail and moves upward first
    pattern = []

    if start_from == "top":
        row = 0
        direction = 1
    elif start_from == "bottom":
        row = num_rails - 1
        direction = -1
    else:
        raise ValueError("'start_from' must be either 'top' or 'bottom'.")

    for _ in range(length):
        pattern.append(row)

        if row == 0:
            direction = 1
        elif row == num_rails - 1:
            direction = -1

        row += direction

    return pattern

### 3. Generate a Random Key (Rail Count)

In [ ]:
def generate_random_key(text_length: int, min_rails: int = 2, max_rails: int = 10) -> int:
    # the key can never be less than 2, or greater than the length of the text
    upper_bound = min(max_rails, text_length)
    return random.randint(min_rails, upper_bound)

### 4. Encryption

In [ ]:
def encrypt(text: str, num_rails: int, start_from: str = "top") -> str:
    validate_printable_text(text)

    if num_rails < 2:
        raise ValueError("'num_rails' must be at least 2.")
    if num_rails > len(text):
        raise ValueError("'num_rails' cannot be greater than the length of the text.")

    pattern = build_rail_pattern(len(text), num_rails, start_from)

    rails = [[] for _ in range(num_rails)]
    for i, ch in enumerate(text):
        rails[pattern[i]].append(ch)

    return "".join("".join(rail) for rail in rails)

### 5. Decryption

In [ ]:
def decrypt(cipher_text: str, num_rails: int, start_from: str = "top") -> str:
    validate_printable_text(cipher_text)

    if num_rails < 2:
        raise ValueError("'num_rails' must be at least 2.")

    length = len(cipher_text)
    pattern = build_rail_pattern(length, num_rails, start_from)

    # how many characters landed on each rail during encryption
    rail_lengths = [pattern.count(rail) for rail in range(num_rails)]

    # slice the ciphertext into rail sized chunks, in the same order they were read out
    rails = []
    position = 0
    for length_of_rail in rail_lengths:
        rails.append(list(cipher_text[position:position + length_of_rail]))
        position += length_of_rail

    # walk the zigzag pattern again, pulling the next unused character from the right rail
    rail_pointers = [0] * num_rails
    plain_text = []
    for rail in pattern:
        plain_text.append(rails[rail][rail_pointers[rail]])
        rail_pointers[rail] += 1

    return "".join(plain_text)

### 6. Verify the Hand Worked Example in Code

In [ ]:
hand_plaintext = "WEAREDISCOVERED"
hand_cipher = encrypt(hand_plaintext, num_rails=3, start_from="top")
hand_decrypted = decrypt(hand_cipher, num_rails=3, start_from="top")

print(f"Plaintext: {hand_plaintext}")
print(f"Encrypted (from top): {hand_cipher}  (should match WECRERDSOEEAIVD from the hand example)")
print(f"Decrypted: {hand_decrypted}")

bottom_cipher = encrypt(hand_plaintext, num_rails=3, start_from="bottom")
bottom_decrypted = decrypt(bottom_cipher, num_rails=3, start_from="bottom")

print(f"\nEncrypted (from bottom): {bottom_cipher}  (different ciphertext, same rail count)")
print(f"Decrypted: {bottom_decrypted}")

Plaintext: WEAREDISCOVERED
Encrypted (from top): WECRERDSOEEAIVD  (should match WECRERDSOEEAIVD from the hand example)
Decrypted: WEAREDISCOVERED

Encrypted (from bottom): AIVDERDSOEEWECR  (different ciphertext, same rail count)
Decrypted: WEAREDISCOVERED


### 7. Example usage

In [ ]:
plaintext = """TOP secret Massage! Agent 101, visit Area 51 (37d14'0\"N 115d48'30\"W)."""

key = generate_random_key(len(plaintext))
print(f"Generated Random Key (rail count): {key}")

Generated Random Key (rail count): 5


In [ ]:
print(f"Original Plain Text: {plaintext}")

cipher_text = encrypt(plaintext, key)
print("Encrypted:", cipher_text)

decrypted_text = decrypt(cipher_text, key)
print("Decrypted:", decrypted_text)

match = plaintext == decrypted_text
print(f"Verification Match:{match}")

Original Plain Text: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Encrypted: Tegtiad10Ortaen vse 71 13"Pc s!e1 ir534N5'W eMs g0,tA1('"d8)saA1  04.
Decrypted: TOP secret Massage! Agent 101, visit Area 51 (37d14'0"N 115d48'30"W).
Verification Match:True


### 8. Trying Different Track Sizes on the Same Message

In [ ]:
for start_from in ["top", "bottom"]:
    print(f"--- Starting from {start_from} ---")
    for rail_count in [2, 3, 4, 5, 8]:
        cipher_text = encrypt(plaintext, rail_count, start_from)
        decrypted_text = decrypt(cipher_text, rail_count, start_from)
        print(f"Rails={rail_count:<2} Cipher: {cipher_text}")
        print(f"          Match: {decrypted_text == plaintext}")
    print()

--- Starting from top ---
Rails=2  Cipher: TPsce asg!Aet11 ii ra5 3d40N154'0W.O ertMsae gn 0,vstAe 1(71'" 1d83")
          Match: True
Rails=3  Cipher: TseagAt1i a d0140.O ertMsae gn 0,vstAe 1(71'" 1d83")Pc s!e1 ir534N5'W
          Match: True
Rails=4  Cipher: Tca!t  5dN4WOerMse n ,vtA 171" d8")Pse sgAe11iira 34015'0. tag0se('13
          Match: True
Rails=5  Cipher: Tegtiad10Ortaen vse 71 13"Pc s!e1 ir534N5'W eMs g0,tA1('"d8)saA1  04.
          Match: True
Rails=8  Cipher: Ts151Osa0, 1 1Pag1 a N5. Me ve("d)s !tir304Wet nsA7'8"ceAei d4'0rgt13
          Match: True

--- Starting from bottom ---
Rails=2  Cipher: O ertMsae gn 0,vstAe 1(71'" 1d83")TPsce asg!Aet11 ii ra5 3d40N154'0W.
          Match: True
Rails=3  Cipher: Pc s!e1 ir534N5'WO ertMsae gn 0,vstAe 1(71'" 1d83")TseagAt1i a d0140.
          Match: True
Rails=4  Cipher:  tag0se('13Pse sgAe11iira 34015'0.OerMse n ,vtA 171" d8")Tca!t  5dN4W
          Match: True
Rails=5  Cipher: saA1  04. eMs g0,tA1('"d8)Pc s!e1 ir534N5'WOrtaen vse 7